In [ ]:
%pip install -q tokenizers


In [ ]:
import re, json, random, os
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, f1_score, confusion_matrix, precision_recall_curve

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
LABELS = ["toxicity", "hate", "harassment", "abuse"]

JIGSAW_PATH = "train.csv"

MAX_LEN = 100
VOCAB_SIZE = 20000
EMBED_DIM = 128
HIDDEN_DIM = 128
NEG_EMBED_DIM = 8         
SCOPE_CHARS = 30          
DROPOUT = 0.3
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 15
PATIENCE = 3
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

N_HARD_NEGATIVES_PER_PATTERN = 1000   

ARTIFACT_DIR = "artifacts"
MODEL_PATH = os.path.join(ARTIFACT_DIR, "toxicity_model_v4.pt")
TOKENIZER_PATH = os.path.join(ARTIFACT_DIR, "bpe_tokenizer.json")
THRESHOLDS_PATH = os.path.join(ARTIFACT_DIR, "thresholds.json")
CONFIG_PATH = os.path.join(ARTIFACT_DIR, "model_config.json")
SUGGESTION_BANK_PATH = os.path.join(ARTIFACT_DIR, "suggestion_bank.json")

os.makedirs(ARTIFACT_DIR, exist_ok=True)


In [ ]:
jigsaw_df = pd.read_csv(JIGSAW_PATH)

jigsaw_df["toxicity"]   = ((jigsaw_df["toxic"] == 1) | (jigsaw_df["severe_toxic"] == 1)).astype(int)
jigsaw_df["hate"]       = jigsaw_df["identity_hate"].astype(int)
jigsaw_df["harassment"] = ((jigsaw_df["insult"] == 1) | (jigsaw_df["threat"] == 1)).astype(int)
jigsaw_df["abuse"]      = ((jigsaw_df["obscene"] == 1) | (jigsaw_df["severe_toxic"] == 1)).astype(int)

jigsaw_df = jigsaw_df.rename(columns={"comment_text": "text"})
jigsaw_df = jigsaw_df[["text"] + LABELS].dropna().drop_duplicates(subset="text")
jigsaw_df["label_mask"] = [[1, 1, 1, 1]] * len(jigsaw_df)
jigsaw_df["source"] = "jigsaw_en"

print("jigsaw rows:", len(jigsaw_df))
print(jigsaw_df[LABELS].mean())


In [ ]:

SLUR_BLOCKLIST = {
    "nigger", "nigga", "niggers", "faggot", "fag", "faggots",
    "chink", "spic", "kike", "tranny", "retard", "paki",
}

def mine_toxic_words(df, label="toxicity", top_k=150, min_docs=25, max_len=15):
    pos_docs = df.loc[df[label] == 1, "text"].str.lower().str.split()
    neg_docs = df.loc[df[label] == 0, "text"].str.lower().str.split()

    pos_doc_freq = Counter()
    for toks in pos_docs:
        pos_doc_freq.update(set(toks))
    neg_doc_freq = Counter()
    for toks in neg_docs:
        neg_doc_freq.update(set(toks))

    scored = []
    for w, c in pos_doc_freq.items():
        if c < min_docs or not w.isalpha() or not (3 <= len(w) <= max_len):
            continue
        if w in SLUR_BLOCKLIST:
            continue
        ratio = c / (neg_doc_freq.get(w, 0) + 1)
        scored.append((ratio, w))
    scored.sort(reverse=True)
    return [w for _, w in scored[:top_k]]


NEGATION_TEMPLATES = [
    "I don't think you're {w} at all.",
    "That's not {w}, honestly.",
    "You're absolutely not {w}.",
    "No one would call that {w}.",
    "I wouldn't say this is {w}.",
    "Not {w}, just direct.",
]
QUOTATION_TEMPLATES = [
    "He called me {w} but I don't think that's fair.",
    "She said I was being {w}, which really hurt.",
    "They kept saying I was {w} in the group chat.",
    "The article was described as {w} by one reviewer, though others disagreed.",
]
SARCASM_TEMPLATES = [
    "Oh sure, because calling someone {w} is SO mature, right?",
    "Wow, real classy calling people {w} like that.",
    "Great job being {w} to everyone, really impressive.",
]
SELF_REFERENCE_TEMPLATES = [
    "I feel so {w} today, I can't focus on anything.",
    "Honestly I've been pretty {w} about this whole situation myself.",
    "Sometimes I think I'm the {w} one in this argument.",
]
RHETORICAL_TEMPLATES = [
    "Would you really call that {w}?",
    "Is it fair to call someone {w} just for disagreeing?",
    "Why do people think this is {w}?",
]

TEMPLATE_GROUPS = {
    "negation": NEGATION_TEMPLATES,
    "quotation": QUOTATION_TEMPLATES,
    "sarcasm": SARCASM_TEMPLATES,
    "self_reference": SELF_REFERENCE_TEMPLATES,
    "rhetorical": RHETORICAL_TEMPLATES,
}


def generate_hard_negatives(df, n_per_pattern=N_HARD_NEGATIVES_PER_PATTERN, seed=SEED):
    rng = random.Random(seed)
    words = mine_toxic_words(df)
    rows, kinds = [], []
    for group_name, templates in TEMPLATE_GROUPS.items():
        for _ in range(n_per_pattern):
            rows.append(rng.choice(templates).format(w=rng.choice(words)))
            kinds.append(f"synthetic_{group_name}")
    out = pd.DataFrame({"text": rows, "hard_negative_type": kinds}).drop_duplicates(subset="text")
    for lbl in LABELS:
        out[lbl] = 0
    out["label_mask"] = [[1, 1, 1, 1]] * len(out)
    out["source"] = "synthetic_hard_negative"
    return out


synthetic_df = generate_hard_negatives(jigsaw_df)
print("synthetic hard negatives:", len(synthetic_df))
print(synthetic_df["hard_negative_type"].value_counts())
synthetic_df.drop(columns=["hard_negative_type"]).head()


In [ ]:
combined_df = pd.concat(
    [jigsaw_df, synthetic_df.drop(columns=["hard_negative_type"])], ignore_index=True
)
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("combined rows:", len(combined_df))
print(combined_df["source"].value_counts())
print(combined_df[LABELS].mean())


In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WS_RE = re.compile(r"\s+")

def clean_text(text):
    text = URL_RE.sub(" <URL> ", text)
    text = WS_RE.sub(" ", text).strip()
    return text

combined_df["clean_text"] = combined_df["text"].astype(str).apply(clean_text)
combined_df[["clean_text", "source"]].head()


In [ ]:
train_df, temp_df = train_test_split(combined_df, train_size=0.70, random_state=SEED)
val_df, test_df = train_test_split(temp_df, train_size=0.5, random_state=SEED)
len(train_df), len(val_df), len(test_df)


In [ ]:
from tokenizers import ByteLevelBPETokenizer

with open("bpe_train_corpus.txt", "w", encoding="utf-8") as f:
    for t in train_df["clean_text"]:
        f.write(t.replace("\n", " ") + "\n")

bpe = ByteLevelBPETokenizer()
bpe.train(
    files=["bpe_train_corpus.txt"],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<PAD>", "<UNK>", "<URL>"],
)
bpe.save(TOKENIZER_PATH)

PAD_ID = bpe.token_to_id("<PAD>")
UNK_ID = bpe.token_to_id("<UNK>")

print("BPE vocab size:", bpe.get_vocab_size())
print("sample encode:", bpe.encode("you're not an idiot at all").tokens)


In [ ]:
NEGATION_CUES_RE = re.compile(
    r"\b(not|never|no|cannot|hardly|barely|without|neither|nor)\b|\w+n['’]t\b",
    re.IGNORECASE,
)

def compute_negation_scope_mask(text, offsets, scope_chars=SCOPE_CHARS):
    scope_ranges = [(m.end(), m.end() + scope_chars) for m in NEGATION_CUES_RE.finditer(text)]
    mask = []
    for (s, _e) in offsets:
        in_scope = any(rs <= s < re_ for rs, re_ in scope_ranges)
        mask.append(1 if in_scope else 0)
    return mask

def encode_with_features(text, max_len=MAX_LEN):
    enc = bpe.encode(text)
    ids = enc.ids[:max_len]
    offsets = enc.offsets[:max_len]
    neg_scope = compute_negation_scope_mask(text, offsets)
    pad_len = max_len - len(ids)
    ids = ids + [PAD_ID] * pad_len
    neg_scope = neg_scope + [0] * pad_len
    return ids, neg_scope

_ids, _neg = encode_with_features("i don't think you're an idiot at all")
list(zip(bpe.encode("i don't think you're an idiot at all").tokens, _neg))


In [ ]:
class ToxicityDataset(Dataset):
    def __init__(self, dframe):
        self.texts = dframe["clean_text"].tolist()
        self.labels = dframe[LABELS].values.astype("float32")
        self.masks = np.stack(dframe["label_mask"].values).astype("float32")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids, neg_scope = encode_with_features(self.texts[idx])
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(neg_scope, dtype=torch.long),
            torch.tensor(self.labels[idx]),
            torch.tensor(self.masks[idx]),
        )

train_ds = ToxicityDataset(train_df)
val_ds   = ToxicityDataset(val_df)
test_ds  = ToxicityDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

# sanity check -- 4 tensors per batch now: ids, neg_scope, labels, mask
xb, negb, yb, mb = next(iter(train_loader))
xb.shape, negb.shape, yb.shape, mb.shape


In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out, pad_mask):
        scores = self.attn(lstm_out).squeeze(-1)
        scores = scores.masked_fill(pad_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)
        return context, weights


class ContextAwareBiLSTMClassifier(nn.Module):
    """BiLSTM + attention, extended with an explicit negation-scope feature
    channel so 'not X' doesn't read the same as 'X'. Still no transformer /
    self-attention-over-the-whole-sequence architecture -- this is a small
    additive feature embedding concatenated to the word embedding before a
    standard recurrent layer."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels,
                 neg_embed_dim=8, dropout=0.3, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.neg_scope_embedding = nn.Embedding(2, neg_embed_dim)  # 0 = outside scope, 1 = inside
        self.lstm = nn.LSTM(embed_dim + neg_embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = Attention(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)

    def _encode_sequence(self, x, neg_scope):
        pad_mask = (x != self.pad_idx).float()
        word_embed = self.embedding(x)
        neg_embed = self.neg_scope_embedding(neg_scope)
        combined = torch.cat([word_embed, neg_embed], dim=-1)
        lstm_out, _ = self.lstm(combined)
        context, attn_weights = self.attention(lstm_out, pad_mask)
        return context, attn_weights

    def forward(self, x, neg_scope, return_attention=False):
        context, attn_weights = self._encode_sequence(x, neg_scope)
        context = self.dropout(context)
        logits = self.fc(context)
        if return_attention:
            return logits, attn_weights
        return logits

    def encode(self, x, neg_scope):
        context, _ = self._encode_sequence(x, neg_scope)
        return context


model = ContextAwareBiLSTMClassifier(
    bpe.get_vocab_size(), EMBED_DIM, HIDDEN_DIM, len(LABELS),
    neg_embed_dim=NEG_EMBED_DIM, dropout=DROPOUT, pad_idx=PAD_ID,
).to(device)
model


In [ ]:
def masked_focal_loss(logits, targets, mask, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
    probs = torch.sigmoid(logits)
    ce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * (1 - p_t).pow(gamma) * ce
    loss = loss * mask
    return loss.sum() / mask.sum().clamp(min=1.0)


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
best_val_macro_f1 = -1.0
patience_counter = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    train_loss = 0.0
    for x, neg, y, m in train_loader:
        x, neg, y, m = x.to(device), neg.to(device), y.to(device), m.to(device)
        optimizer.zero_grad()
        loss = masked_focal_loss(model(x, neg), y, m)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_probs, val_true = [], []
    with torch.no_grad():
        for x, neg, y, m in val_loader:
            x, neg = x.to(device), neg.to(device)
            val_probs.append(torch.sigmoid(model(x, neg)).cpu())
            val_true.append(y)
    val_probs = torch.cat(val_probs).numpy()
    val_true = torch.cat(val_true).numpy()
    val_preds = (val_probs >= 0.5).astype(int)
    val_macro_f1 = f1_score(val_true, val_preds, average="macro", zero_division=0)

    print(f"epoch {epoch+1}: train_loss={train_loss:.4f} val_macro_f1={val_macro_f1:.4f}")

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(torch.load(MODEL_PATH))
print("Best val macro F1:", best_val_macro_f1)


In [ ]:
def get_probs(loader):
    model.eval()
    all_probs, all_true, all_masks = [], [], []
    with torch.no_grad():
        for x, neg, y, m in loader:
            x, neg = x.to(device), neg.to(device)
            probs = torch.sigmoid(model(x, neg)).cpu()
            all_probs.append(probs)
            all_true.append(y)
            all_masks.append(m)
    return torch.cat(all_probs).numpy(), torch.cat(all_true).numpy(), torch.cat(all_masks).numpy()

val_probs, val_true, val_mask = get_probs(val_loader)

thresholds = {}
for i, label in enumerate(LABELS):
    valid = val_mask[:, i] == 1
    p, r, t = precision_recall_curve(val_true[valid, i], val_probs[valid, i])
    f1 = 2 * p * r / (p + r + 1e-9)
    best_idx = f1[:-1].argmax() if len(t) > 0 else 0
    thresholds[label] = float(t[best_idx]) if len(t) > 0 else 0.5

with open(THRESHOLDS_PATH, "w") as f:
    json.dump(thresholds, f)

thresholds


In [ ]:
test_probs, test_true, test_mask = get_probs(test_loader)

for i, label in enumerate(LABELS):
    valid = test_mask[:, i] == 1
    preds = (test_probs[valid, i] >= thresholds[label]).astype(int)
    p, r, f, _ = precision_recall_fscore_support(test_true[valid, i], preds, average="binary", zero_division=0)
    cm = confusion_matrix(test_true[valid, i], preds)
    fn = int(cm[1][0]) if cm.shape == (2, 2) else None
    print(f"{label:12s} n={valid.sum():>7d} thr={thresholds[label]:.2f} precision={p:.3f} recall={r:.3f} f1={f:.3f} false_negatives={fn}")

all_preds = np.stack([(test_probs[:, i] >= thresholds[label]).astype(int) for i, label in enumerate(LABELS)], axis=1)
print("\ntest macro F1:", f1_score(test_true, all_preds, average="macro", zero_division=0))


In [ ]:
def predict(text, top_n=5):
    cleaned = clean_text(text)
    ids, neg_scope = encode_with_features(cleaned)
    x = torch.tensor([ids], dtype=torch.long).to(device)
    neg = torch.tensor([neg_scope], dtype=torch.long).to(device)

    model.eval()
    with torch.no_grad():
        logits, attn = model(x, neg, return_attention=True)
        probs = torch.sigmoid(logits)[0].cpu().tolist()

    per_label = {label: {"prob": round(p, 4), "flagged": bool(p >= thresholds[label])}
                 for label, p in zip(LABELS, probs)}
    overall = 1 - float(np.prod([1 - p for p in probs]))

    tokens = bpe.encode(cleaned).tokens[:MAX_LEN]
    weights = attn[0][: len(tokens)].cpu().tolist()
    top_tokens = sorted(zip(tokens, weights), key=lambda t: -t[1])[:top_n]

    return {
        "per_label": per_label,
        "overall_toxicity": round(overall, 4),
        "any_flagged": any(v["flagged"] for v in per_label.values()),
        "top_attended_tokens": [{"token": t, "weight": round(w, 4)} for t, w in top_tokens],
    }

for example in [
    "I really enjoyed this movie.",
    "You are a disgusting idiot.",
    "I don't think you're an idiot.",
    "xoxo ur so dum lol",
]:
    print(example, "->", predict(example))


In [ ]:
SUGGESTION_BANK = [
    {"category": "toxicity", "trigger_context": "You are so stupid, this makes no sense.",
     "suggested_rewrite": "I'm having trouble following this - could you walk me through your reasoning?"},
    {"category": "toxicity", "trigger_context": "This is the dumbest thing I've ever read.",
     "suggested_rewrite": "I disagree with this and think it has some significant flaws - here's why."},
    {"category": "toxicity", "trigger_context": "What an idiot, how did you mess this up so badly.",
     "suggested_rewrite": "This didn't turn out the way I expected - let's figure out what went wrong."},
    {"category": "toxicity", "trigger_context": "Your opinion is garbage and so are you.",
     "suggested_rewrite": "I see this very differently - here's where I think we disagree."},

    {"category": "harassment", "trigger_context": "Stop replying to my posts, nobody wants you here.",
     "suggested_rewrite": "I'd prefer we don't continue this conversation - let's leave it here."},
    {"category": "harassment", "trigger_context": "I'm going to make sure everyone knows what a loser you are.",
     "suggested_rewrite": "I strongly disagree with you and don't think we see eye to eye on this."},
    {"category": "harassment", "trigger_context": "Keep pushing me and see what happens to you.",
     "suggested_rewrite": "I'm getting frustrated - I think we should both step back from this thread."},

    {"category": "hate", "trigger_context": "People like you don't belong here.",
     "suggested_rewrite": "I think we have very different perspectives, shaped by different backgrounds."},
    {"category": "hate", "trigger_context": "Your kind always causes problems.",
     "suggested_rewrite": "I've had frustrating experiences in situations like this before."},

    {"category": "abuse", "trigger_context": "Shut up, you worthless piece of garbage.",
     "suggested_rewrite": "I need a moment - can we pause this conversation?"},
    {"category": "abuse", "trigger_context": "You disgust me, get lost.",
     "suggested_rewrite": "I'm upset right now and would like some space."},
]

GUIDELINE_NOTE = {
    "toxicity":   "Community Guidelines: keep feedback focused on ideas, not people.",
    "hate":       "Community Guidelines: content targeting people based on identity is not allowed.",
    "harassment": "Community Guidelines: repeated targeting or threats toward a person are not allowed.",
    "abuse":      "Community Guidelines: personal insults and degrading language are not allowed.",
}

# most severe first -- decides which category's suggestion wins when several are flagged
SEVERITY_ORDER = ["hate", "harassment", "abuse", "toxicity"]


def embed_text(text):
    cleaned = clean_text(text)
    ids, neg_scope = encode_with_features(cleaned)
    x = torch.tensor([ids], dtype=torch.long).to(device)
    neg = torch.tensor([neg_scope], dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        vec = model.encode(x, neg)[0].cpu().numpy()
    return vec


def cosine_sim_matrix(query, bank_matrix):
    q = query / (np.linalg.norm(query) + 1e-9)
    b = bank_matrix / (np.linalg.norm(bank_matrix, axis=1, keepdims=True) + 1e-9)
    return b @ q


SUGGESTION_BANK_EMBEDDINGS = np.stack([embed_text(e["trigger_context"]) for e in SUGGESTION_BANK])


def suggest_alternative(text, per_label):
    flagged_categories = [c for c in SEVERITY_ORDER if per_label.get(c, {}).get("flagged")]
    if not flagged_categories:
        return None
    primary_category = flagged_categories[0]

    candidate_idxs = [i for i, e in enumerate(SUGGESTION_BANK) if e["category"] == primary_category]
    if not candidate_idxs:
        candidate_idxs = list(range(len(SUGGESTION_BANK)))

    query_vec = embed_text(text)
    sims = cosine_sim_matrix(query_vec, SUGGESTION_BANK_EMBEDDINGS[candidate_idxs])
    best_local = int(np.argmax(sims))
    best_idx = candidate_idxs[best_local]
    best_entry = SUGGESTION_BANK[best_idx]

    return {
        "matched_category": primary_category,
        "similarity": round(float(sims[best_local]), 4),
        "suggested_rewrite": best_entry["suggested_rewrite"],
        "guideline_note": GUIDELINE_NOTE[primary_category],
    }


In [ ]:
def get_toxicity_level(toxicity_score):
    if toxicity_score < 0.60:
        return {"level": "safe", "blur": 0, "message": "No blur required."}
    elif toxicity_score < 0.85:
        return {"level": "moderate", "blur": 1, "message": "Content temporarily blurred due to potentially harmful language."}
    else:
        return {"level": "high", "blur": 2, "message": "Content heavily blurred due to highly toxic or harmful language."}


def get_toxicity_warning(toxicity_score):
    if toxicity_score < 0.60:
        return {"show_warning": False, "title": "Content appears safe",
                "reason": "The detected toxicity level is below the warning threshold."}
    elif toxicity_score < 0.85:
        return {"show_warning": True, "title": "Potentially harmful content",
                "reason": "This content may contain offensive, abusive, or inappropriate language. It has been temporarily blurred."}
    else:
        return {"show_warning": True, "title": "Highly toxic content",
                "reason": "This content has a high probability of containing severely offensive, abusive, threatening, or harmful language. It has been heavily blurred for safety."}


def get_age_rating(toxicity_score):
    if toxicity_score < 0.60:
        return {"age_rating": "13-15", "level": "Low"}
    elif toxicity_score < 0.70:
        return {"age_rating": "15-18", "level": "Moderate"}
    elif toxicity_score < 0.85:
        return {"age_rating": "18-21", "level": "High"}
    else:
        return {"age_rating": "21+", "level": "Very High"}


def build_rating_payload(toxicity_score):
    tox_level = get_toxicity_level(toxicity_score)
    warning = get_toxicity_warning(toxicity_score)
    age = get_age_rating(toxicity_score)
    return {
        "ageRating": age["age_rating"],
        "is_sensitive": warning["show_warning"],
        "toxicity_rating": tox_level["level"],
        "message": tox_level["message"],
        "blur_level": tox_level["blur"],
        "warning_title": warning["title"],
        "warning_reason": warning["reason"],
        "age_rating_level": age["level"],
    }


def moderate(text):
    result = predict(text)
    rating = build_rating_payload(result["overall_toxicity"])
    suggestion = suggest_alternative(text, result["per_label"]) if result["any_flagged"] else None
    return {
        "input_text": text,
        **result,
        "rating": rating,
        "suggestion": suggestion,
    }


for example in [
    "I really enjoyed this movie.",
    "You are a disgusting idiot.",
    "I don't think you're an idiot.",
    "Stop replying to my posts, nobody wants you here.",
]:
    print(json.dumps(moderate(example), indent=2))
    print()


In [ ]:
with open(CONFIG_PATH, "w") as f:
    json.dump({
        "vocab_size": bpe.get_vocab_size(), "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM,
        "neg_embed_dim": NEG_EMBED_DIM, "scope_chars": SCOPE_CHARS,
        "num_labels": len(LABELS), "dropout": DROPOUT, "max_len": MAX_LEN, "labels": LABELS,
        "focal_alpha": FOCAL_ALPHA, "focal_gamma": FOCAL_GAMMA, "pad_id": PAD_ID,
    }, f, indent=2)

with open(SUGGESTION_BANK_PATH, "w") as f:
    json.dump(SUGGESTION_BANK, f, indent=2)  

print("Saved to", ARTIFACT_DIR + "/:")
for fname in [MODEL_PATH, TOKENIZER_PATH, THRESHOLDS_PATH, CONFIG_PATH, SUGGESTION_BANK_PATH]:
    print(" -", fname)
